In [1]:
from pyspark.sql import SparkSession
import getpass
username = getpass.getuser()

spark = SparkSession. \
builder. \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/{username}/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [2]:
orders_df = spark.read.parquet("data/external_table/part-00000-6109de51-a0e8-41af-839d-58e5ef769898-c000.snappy.parquet", inferSchema=True, header=True)

In [3]:
orders_df.show(10)

+--------+-------------------+-----------+---------------+
|order_id|         order_date|customer_id|   order_status|
+--------+-------------------+-----------+---------------+
|       1|2013-07-25 00:00:00|      11599|         CLOSED|
|       2|2013-07-25 00:00:00|        256|PENDING_PAYMENT|
|       3|2013-07-25 00:00:00|      12111|       COMPLETE|
|       4|2013-07-25 00:00:00|       8827|         CLOSED|
|       5|2013-07-25 00:00:00|      11318|       COMPLETE|
|       6|2013-07-25 00:00:00|       7130|       COMPLETE|
|       7|2013-07-25 00:00:00|       4530|       COMPLETE|
|       8|2013-07-25 00:00:00|       2911|     PROCESSING|
|       9|2013-07-25 00:00:00|       5657|PENDING_PAYMENT|
|      10|2013-07-25 00:00:00|       5648|PENDING_PAYMENT|
+--------+-------------------+-----------+---------------+
only showing top 10 rows



In [4]:
orders_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_status: string (nullable = true)



In [5]:
orders_df.schema

StructType(List(StructField(order_id,IntegerType,true),StructField(order_date,TimestampType,true),StructField(customer_id,IntegerType,true),StructField(order_status,StringType,true)))

In [6]:
orders_df.columns

['order_id', 'order_date', 'customer_id', 'order_status']

In [7]:
orders_df.dtypes

[('order_id', 'int'),
 ('order_date', 'timestamp'),
 ('customer_id', 'int'),
 ('order_status', 'string')]

In [8]:
orders_df.createOrReplaceTempView('orders')

### 1. Customers with most number of 'CLOSED' orders

In [10]:
spark.sql("""
            SELECT customer_id, COUNT(order_id) AS order_closed_count 
            FROM orders 
            WHERE order_status='CLOSED' 
            GROUP BY customer_id 
            ORDER BY order_closed_count DESC
            LIMIT 10
""")

customer_id,order_closed_count
1833,6
1363,5
1687,5
5493,5
2403,4
7850,4
10018,4
2236,4
2768,4
10111,4


In [12]:
order_agg = orders_df.filter("order_status = 'CLOSED'").groupBy('customer_id').count()

In [13]:
order_agg.show(10)

+-----------+-----+
|customer_id|count|
+-----------+-----+
|       9900|    1|
|       6397|    2|
|      12027|    1|
|        496|    1|
|       1591|    2|
|       2866|    1|
|        148|    2|
|       6466|    3|
|      11317|    1|
|       6357|    1|
+-----------+-----+
only showing top 10 rows



In [14]:
order_agg.sort('count', ascending=False).show(10)

+-----------+-----+
|customer_id|count|
+-----------+-----+
|       1833|    6|
|       1687|    5|
|       1363|    5|
|       5493|    5|
|       3631|    4|
|       7850|    4|
|       2768|    4|
|       7879|    4|
|      10018|    4|
|       2774|    4|
+-----------+-----+
only showing top 10 rows



In [15]:
spark.stop()

Notes:
If GROUP BY is used, then ORDER BY can be done on only two things:

    Columns that are part of the GROUP BY clause (like customer_id).
    Aggregate functions (like COUNT, SUM, or AVG).